In [25]:
import pandas as pd
import re

# -----------------------------
# 0) Load
# -----------------------------
ing_db = pd.read_csv('../flavordb.csv', index_col=0)
molecule_db = pd.read_csv('../molecules.csv', index_col=0)

ing_db = ing_db.rename(columns={"entity id": "entity_id"})
molecule_db = molecule_db.rename(columns={"pubchem id": "molecule_id",
                                          "common name": "mol_name",
                                          "flavor profile": "flavor_profile"})

train = pd.read_csv('./train_recipes_sauce_full_vocab.csv')  # 경로 필요하면 수정

# -----------------------------
# 1) Choose column name in train
# -----------------------------
# 혹시 ingredient 컬럼명이 다를 수 있어서 자동 탐색
cand_cols = [c for c in train.columns if c.lower() in ["ingredient", "ingredients"]]
if not cand_cols:
    raise ValueError(f"train_recipes_sauce_full_vocab.csv에 ingredient 컬럼이 안 보입니다. 컬럼들: {list(train.columns)}")
ing_col = cand_cols[0]

# -----------------------------
# 2) Normalization helper
# -----------------------------
def norm(s: str) -> str:
    if pd.isna(s):
        return ""
    s = str(s).strip().lower()
    # 공백/구두점 정리 (원하는 규칙 있으면 여기만 바꾸면 됨)
    s = re.sub(r"[_\-]+", " ", s)          # _,- -> space
    s = re.sub(r"\s+", " ", s)            # multiple spaces -> single
    s = re.sub(r"[^\w\s']", "", s)        # punctuation 제거(단, '는 남김)
    return s.strip()

# -----------------------------
# 3) Build sets
# -----------------------------
alias_raw = ing_db["alias"].dropna().astype(str)
alias_set = set(alias_raw.map(norm))

train_raw = train[ing_col].dropna().astype(str)
train_norm = train_raw.map(norm)

train_set = set(train_norm)

# -----------------------------
# 4) Compare
# -----------------------------
inter = alias_set & train_set
only_in_train = train_set - alias_set
only_in_alias = alias_set - train_set

print("=== BASIC STATS ===")
print(f"#alias(unique, normalized): {len(alias_set):,}")
print(f"#train ingredient(unique, normalized): {len(train_set):,}")
print(f"#intersection: {len(inter):,}")

print("\n=== MISMATCH COUNTS ===")
print(f"train에만 있는 항목(=alias에서 못 찾음): {len(only_in_train):,}  ({len(only_in_train)/max(1,len(train_set)):.1%} of train uniques)")
print(f"alias에만 있는 항목(=train에 없음): {len(only_in_alias):,}  ({len(only_in_alias)/max(1,len(alias_set)):.1%} of alias uniques)")

# -----------------------------
# 5) train 쪽 미스매치: 빈도 기준 Top-N
# -----------------------------
# (train 데이터가 "행마다 ingredient"인지, "레시피에 여러 재료가 리스트/문자열로 들어있는지"에 따라 전처리가 다를 수 있음.
#  우선 ingredient 컬럼이 단일 이름이라고 가정하고 빈도 출력.)
miss_mask = ~train_norm.isin(alias_set)
miss_counts = train_raw[miss_mask].value_counts().head(30)

print("\n=== TOP-30 train items NOT IN alias (raw strings) ===")
print(miss_counts)

# -----------------------------
# 6) 예시 몇 개 보기 (정규화된 차이 확인용)
# -----------------------------
print("\n=== SAMPLE (normalized) only_in_train ===")
print(list(sorted(only_in_train))[:30])

print("\n=== SAMPLE (normalized) only_in_alias ===")
print(list(sorted(only_in_alias))[:30])


=== BASIC STATS ===
#alias(unique, normalized): 935
#train ingredient(unique, normalized): 3,433
#intersection: 396

=== MISMATCH COUNTS ===
train에만 있는 항목(=alias에서 못 찾음): 3,037  (88.5% of train uniques)
alias에만 있는 항목(=train에 없음): 539  (57.6% of alias uniques)

=== TOP-30 train items NOT IN alias (raw strings) ===
ingredient
pork loin steak         1
olive oil               1
lemon juice             1
cilantro                1
oil                     1
soy sauce               1
egg                     1
mayonnaise              1
lime juice              1
garlic powder           1
green onion             1
sour cream              1
chicken breast          1
broth                   1
cornstarch              1
paprika                 1
scallion                1
sesame oil              1
cayenne pepper          1
orange juice            1
tomato paste            1
salsa                   1
chili powder            1
worcestershire sauce    1
green pepper            1
red pepper              

Vocab.csv에 없는 flavordb.csv 재료 체크

In [26]:
import pandas as pd
import re

# -----------------------------
# Load
# -----------------------------
ing_db = pd.read_csv('../flavordb.csv', index_col=0).rename(columns={"entity id": "entity_id"})
vocab = pd.read_csv('../vocab.csv')   # <= vocab.csv로 변경

# ingredient 컬럼 자동 탐색 (vocab.csv도 동일 로직 사용)
cand_cols = [c for c in vocab.columns if c.lower() in ["ingredient", "ingredients"]]
if not cand_cols:
    raise ValueError(f"vocab.csv에 ingredient 컬럼이 안 보입니다. 컬럼들: {list(vocab.columns)}")
ing_col = cand_cols[0]

# -----------------------------
# Normalization
# -----------------------------
def norm(s: str) -> str:
    if pd.isna(s):
        return ""
    s = str(s).strip().lower()
    s = re.sub(r"[_\-]+", " ", s)
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"[^\w\s']", "", s)
    return s.strip()

# -----------------------------
# Build vocab sets
# -----------------------------
alias_raw = ing_db["alias"].dropna().astype(str)
alias_norm = alias_raw.map(norm)
alias_set = set(alias_norm)

vocab_raw = vocab[ing_col].dropna().astype(str)
vocab_norm = vocab_raw.map(norm)
vocab_set = set(vocab_norm)

# -----------------------------
# alias에는 있지만 vocab에는 없는 것들
# -----------------------------
alias_only_norm = sorted(alias_set - vocab_set)

print(f"# alias unique: {len(alias_set)}")
print(f"# vocab unique: {len(vocab_set)}")
print(f"# alias ∖ vocab: {len(alias_only_norm)}")

# -----------------------------
# 보기 좋게 DataFrame으로
# -----------------------------
alias_only_df = (
    pd.DataFrame({"alias_norm": alias_only_norm})
      .merge(
          pd.DataFrame({"alias_raw": alias_raw, "alias_norm": alias_norm}),
          on="alias_norm",
          how="left"
      )
      .drop_duplicates()
      .sort_values("alias_norm")
)

display(alias_only_df)

# -----------------------------
# Save alias ∖ vocab to CSV
# -----------------------------
out_path = "./alias_not_in_vocab.csv"
alias_only_df.to_csv(out_path, index=False, encoding="utf-8")

print(f"Saved {len(alias_only_df)} rows to {out_path}")


# alias unique: 935
# vocab unique: 10778
# alias ∖ vocab: 387


,alias_norm,alias_raw
0,abiyuch,abiyuch
1,acerola,acerola
2,achilleas,achilleas
3,akutaq,akutaq
4,alaska blackfish,alaska blackfish
...,...,...
382,yellow passionfruit,yellow passionfruit
383,yellow pond lily,yellow pond lily
384,yellow zucchini,yellow zucchini
385,yellowtail amberjack,yellowtail amberjack


Saved 387 rows to ./alias_not_in_vocab.csv


수작업으로 매핑 후 flavordb 재료명 변경

In [27]:
import pandas as pd
import re
from pathlib import Path

# -----------------------------
# Normalization (same as yours)
# -----------------------------
def norm(s: str) -> str:
    if pd.isna(s):
        return ""
    s = str(s).strip().lower()
    s = re.sub(r"[_\-]+", " ", s)
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"[^\w\s']", "", s)
    return s.strip()

# -----------------------------
# Paths
# -----------------------------
FLAVORDB_PATH = Path("../flavordb.csv")
ALIAS_FILLED_PATH = Path("./alias_not_in_vocab_filled.csv")
OUT_PATH = Path("./flavordb_alias_renamed.csv")

# -----------------------------
# Load
# -----------------------------
df = pd.read_csv(FLAVORDB_PATH)  # index_col=0 안 줘도 됨 (첫 컬럼이 비어있을 수 있어서)
filled = pd.read_csv(ALIAS_FILLED_PATH)

# 컬럼명 방어적으로 찾기
def find_col(df, name):
    m = {c.strip().lower(): c for c in df.columns}
    key = name.strip().lower()
    if key not in m:
        raise ValueError(f"Column '{name}' not found. Available: {list(df.columns)}")
    return m[key]

col_ing = find_col(filled, "Ing Name")
col_new1 = find_col(filled, "New Name 1")

# -----------------------------
# Build mapping: Ing Name -> New Name 1 (non-empty only)
# -----------------------------
tmp = filled[[col_ing, col_new1]].copy()
tmp[col_ing] = tmp[col_ing].astype(str)
tmp[col_new1] = tmp[col_new1].fillna("").astype(str).map(lambda x: x.strip())

tmp["ing_norm"] = tmp[col_ing].map(norm)
tmp["new1"] = tmp[col_new1]

# New Name 1 없는 row 제외
tmp = tmp[tmp["new1"] != ""].copy()

# (선택) 사실상 변경 없는 경우까지 제외하고 싶으면 이 줄 ON
# tmp = tmp[tmp["ing_norm"] != tmp["new1"].map(norm)].copy()

rename_map = dict(zip(tmp["ing_norm"], tmp["new1"]))
print(f"[INFO] mapping size (non-empty New Name 1): {len(rename_map)}")

# -----------------------------
# Apply rename to df['alias']
# -----------------------------
if "alias" not in df.columns:
    raise ValueError(f"'alias' column not found. Available: {list(df.columns)}")

before = df["alias"].astype(str)
df["alias_norm"] = df["alias"].map(norm)

# alias_norm이 매핑키에 있으면 new1으로 교체
df["alias"] = df.apply(
    lambda r: rename_map.get(r["alias_norm"], r["alias"]),
    axis=1
)

n_changed = (before != df["alias"].astype(str)).sum()
print(f"[DONE] alias changed rows: {n_changed}")

# 중간 컬럼 제거하고 저장
df = df.drop(columns=["alias_norm"])
df.to_csv(OUT_PATH, index=False, encoding="utf-8")
print(f"[DONE] Saved: {OUT_PATH}")


[INFO] mapping size (non-empty New Name 1): 314
[DONE] alias changed rows: 313
[DONE] Saved: flavordb_alias_renamed.csv


In [28]:
df = pd.read_csv("./flavordb_alias_renamed.csv")

print("shape:", df.shape)
display(df.head(5))

shape: (935, 7)


,Unnamed: 0,entity id,alias,synonyms,scientific name,category,molecules
0,0,1,bread,{'bakery products'},poacceae,bakery,"{27457, 31252, 7976, 22201, 26331, 26808}"
1,1,2,bread,{'bread'},poacceae,bakery,"{1031, 1032, 644104, 527, 8723, 31260, 15394, ..."
2,2,3,rye bread,{'rye bread'},rye,bakery,"{32065, 644104, 72, 18635, 460, 332, 12366, 89..."
3,3,4,bread,"{'soda scones', 'soda farls'}",wheat,bakery,"{30914, 6915, 5365891, 12170, 14286, 8082, 312..."
4,4,5,white bread,{'white bread'},wheat,bakery,"{7361, 994, 10883, 7362, 11173, 5365891, 11559..."


Drop FlavorDB-only ingredients

In [29]:
import pandas as pd
import re

# -----------------------------
# Load
# -----------------------------
ing_db = pd.read_csv("./flavordb_alias_renamed.csv")
vocab = pd.read_csv("../vocab.csv")

# ingredient 컬럼 자동 탐색
cand_cols = [c for c in vocab.columns if c.lower() in ["ingredient", "ingredients"]]
if not cand_cols:
    raise ValueError(f"vocab.csv에 ingredient 컬럼이 안 보입니다. 컬럼들: {list(vocab.columns)}")
ing_col = cand_cols[0]

# -----------------------------
# Normalization
# -----------------------------
def norm(s: str) -> str:
    if pd.isna(s):
        return ""
    s = str(s).strip().lower()
    s = re.sub(r"[_\-]+", " ", s)
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"[^\w\s']", "", s)
    return s.strip()

# -----------------------------
# Build vocab set
# -----------------------------
vocab_norm = (
    vocab[ing_col]
    .dropna()
    .astype(str)
    .map(norm)
)
vocab_set = set(vocab_norm)

# -----------------------------
# Filter flavordb: keep alias ∈ vocab
# -----------------------------
if "alias" not in ing_db.columns:
    raise ValueError(f"'alias' 컬럼이 없습니다. 컬럼들: {list(ing_db.columns)}")

ing_db["alias_norm"] = ing_db["alias"].map(norm)

mask_keep = ing_db["alias_norm"].isin(vocab_set)

filtered = ing_db[mask_keep].copy()

print(f"# original rows: {len(ing_db)}")
print(f"# kept rows (alias ∈ vocab): {len(filtered)}")
print(f"# removed rows: {len(ing_db) - len(filtered)}")

# -----------------------------
# Drop helper column and save
# -----------------------------
filtered = filtered.drop(columns=["alias_norm"])

out_path = "./flavordb_alias_in_vocab_only.csv"
filtered.to_csv(out_path, index=False, encoding="utf-8")

print(f"[DONE] Saved filtered flavordb to {out_path}")

# -----------------------------
# Sanity check: alias ∖ vocab == 0 ?
# -----------------------------
alias_after = set(filtered["alias"].dropna().astype(str).map(norm))
alias_minus_vocab = alias_after - vocab_set

print(f"[CHECK] alias ∖ vocab after filtering: {len(alias_minus_vocab)}")
assert len(alias_minus_vocab) == 0, "❌ alias ∖ vocab 이 0이 아닙니다!"
print("✅ alias ∖ vocab == 0 confirmed")


# original rows: 935
# kept rows (alias ∈ vocab): 857
# removed rows: 78
[DONE] Saved filtered flavordb to ./flavordb_alias_in_vocab_only.csv
[CHECK] alias ∖ vocab after filtering: 0
✅ alias ∖ vocab == 0 confirmed


In [30]:
df = pd.read_csv("./flavordb_alias_in_vocab_only.csv")

print("shape:", df.shape)
display(df.head(5))

shape: (857, 7)


,Unnamed: 0,entity id,alias,synonyms,scientific name,category,molecules
0,0,1,bread,{'bakery products'},poacceae,bakery,"{27457, 31252, 7976, 22201, 26331, 26808}"
1,1,2,bread,{'bread'},poacceae,bakery,"{1031, 1032, 644104, 527, 8723, 31260, 15394, ..."
2,2,3,rye bread,{'rye bread'},rye,bakery,"{32065, 644104, 72, 18635, 460, 332, 12366, 89..."
3,3,4,bread,"{'soda scones', 'soda farls'}",wheat,bakery,"{30914, 6915, 5365891, 12170, 14286, 8082, 312..."
4,4,5,white bread,{'white bread'},wheat,bakery,"{7361, 994, 10883, 7362, 11173, 5365891, 11559..."
